# EXEMPLO 1 — Multi-Query RAG Mínimo
## Referência Rápida · Aula 7

Este notebook demonstra o Multi-Query RAG em 5 células simples.
Use como referência antes de implementar o LAB1 completo.

In [ ]:
!pip install -q langchain langchain-openai sentence-transformers
print('✅ Instalado!')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# LLM via vLLM
llm = ChatOpenAI(
    base_url='http://localhost:8000/v1',
    api_key='dummy',
    model='meta-llama/Llama-3.1-8B-Instruct',
    temperature=0.7
)

# Prompt de variações
PROMPT = PromptTemplate(
    input_variables=['question'],
    template="""Gere 4 variações jurídicas da pergunta abaixo. Retorne uma por linha:
{question}
Variações:"""
)

print('✅ LLM configurado!')

In [ ]:
# Gerar variações
query = 'Quando posso ser preso preventivamente?'
response = llm.invoke(PROMPT.format(question=query))
variacoes = [v.strip() for v in response.content.strip().split('\n') if v.strip()]

print(f'Query original: {query}')
print('\nVariações:')
for i, v in enumerate(variacoes, 1):
    print(f'  {i}. {v}')

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Deduplicação simples
model = SentenceTransformer('BAAI/bge-m3')

def deduplicate(docs, threshold=0.85):
    if not docs:
        return docs
    embs = model.encode([d['content'] for d in docs], normalize_embeddings=True)
    selecionados = [0]
    for i in range(1, len(docs)):
        if max(np.dot(embs[i], embs[j]) for j in selecionados) < threshold:
            selecionados.append(i)
    return [docs[i] for i in selecionados]

# Exemplo com docs fictícios
docs_exemplo = [
    {'id': 'A', 'content': 'A prisão preventiva é cabível nos casos do art. 312 do CPP.'},
    {'id': 'B', 'content': 'A prisão cautelar é decretada quando necessária para a investigação criminal.'},
    {'id': 'C', 'content': 'As medidas cautelares alternativas devem ser preferidas à prisão preventiva.'},
    {'id': 'D', 'content': 'A prisão preventiva requer pressupostos legais expressos no CPP art. 312.'}  # quase igual a A
]

docs_unicos = deduplicate(docs_exemplo)
print(f'Docs antes: {len(docs_exemplo)} | Docs após dedup: {len(docs_unicos)}')

In [ ]:
print('✅ EXEMPLO 1 concluído!')
print(f'Multi-Query em {4} células principais:')
print('  1. Instalar dependências')
print('  2. Configurar LLM + prompt')
print('  3. Gerar variações')
print('  4. Deduplicar por similaridade')
print('\n→ Para a implementação completa, ver LAB1_Multi_Query_RAG.ipynb')